In [1]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matlab.engine
eng = matlab.engine.start_matlab()
eng.addpath('dual-phase-slope', nargout=0)

In [16]:
fdf = load_and_preprocess_fnirs_data()
fdf, fdf_all = extract_oxy_deoxy(fdf)
print(fdf.head(10))

x = len(fdf_all)

for i, trial in enumerate(fdf_all):
    print(trial["pid"])
    trial.to_csv('{}_processed_trial{}.csv'.format(trial["pid"][0], i+1))

trial starts [180, 1417, 3370]
trial ends [886, 2558, 4553]
trial starts [205, 2017, 3494, 6976]
trial ends [1475, 2921, 4457, 6837, 7740]
trial starts [163, 1955, 3852, 5572]
trial ends [1360, 3323, 4882, 6557]
trial starts [256, 3037, 4699]
trial ends [2380, 4037, 6165]
trial starts [165, 2173, 4612]
trial ends [1580, 3412, 5780]
trial starts [177, 1793, 4139]
trial ends [1258, 3050, 6228]
trial starts [220, 1906, 4104, 5707]
trial ends [1503, 3345, 4716, 8144]
trial starts [183, 2227, 4362, 6399]
trial ends [1740, 3711, 5462, 7389]
trial starts [184, 1619, 3649]
trial ends [1171, 2847, 4618]
trial starts [369, 3782, 5709]
trial ends [2098, 5290, 7005]
trial starts [154, 2072, 4028]
trial ends [1538, 3160, 5185]
trial starts [55, 1899, 4055]
trial ends [1174, 3008, 5717]
trial starts [210, 1898, 3810, 6127]
trial ends [1216, 1222, 2930, 4954, 8027]
trial starts [158, 1895, 3750]
trial ends [1184, 2565, 5042]
trial starts [16, 2788, 4748]
trial ends [1195, 3918, 6676]
trial starts [20

In [2]:
import math 

fs = 5.2
seconds_to_cut_from_either_end = 5

def extract_task_dfs(df):
    """ 
        For each participant, extracts their individual task dfs            
    """
    pdatas = {}
    for pid, pdf in df.groupby('pid'):
        pdatas[pid] = []

        pdf = pdf.copy().reset_index(drop=True)
        
        start_trial_indices = list(pdf[pdf['MARKER'].str.contains("BEGIN")].index.values) 
        start_trial_indices = [x-int(fs*seconds_to_cut_from_either_end) for x in start_trial_indices]
        
        start_form_indices = list(pdf[pdf['MARKER'].str.contains("END")].index.values) 
        start_form_indices = [x+int(fs*seconds_to_cut_from_either_end) for x in start_form_indices]
        
        print("trial starts", start_trial_indices)
        print("trial ends", start_form_indices)


        # end_trial_indices include also the start of the second trial for each pair
        end_trial_indices = sorted(start_trial_indices[1:] + start_form_indices + [len(pdf)])

        for i, start_idx in enumerate(start_trial_indices):

            # because end_trial_indices also include starts, only get the first end index that is greater than the start index           
            end_idx = next((ep for ep in end_trial_indices if ep > start_idx))
            
            task_df = pdf.iloc[start_idx:end_idx].copy().reset_index(drop=True)  
            
            # NOTE: we need 'something' here to align the qualtrics and fnirs data. 
            #       but print the pid and set this fnirs data aside later to be skipped!
            if task_df.shape[0] < fs * seconds_to_cut_from_either_end * 2:
                task_df = task_df.copy().reset_index(drop=True)
                print(f"TOO FEW DATAPOINTS FOR - pid: {pid} - {i} - {task_df.shape}")
            else:            
                # remove the first and last n seconds of data
                task_df = task_df.iloc[math.ceil(fs * seconds_to_cut_from_either_end) : 
                                      -math.ceil(fs * seconds_to_cut_from_either_end)].reset_index(drop=True)
            
            pdatas[pid].append(task_df)

    return pdatas



def extract_oxy_deoxy(df):
        
    # extract the oxy and deoxy values for each trial
    all_segments = []

    pdatas = extract_task_dfs(df)

    for pid, pdata_list in pdatas.items():
        # print(pid)
        all_pdata = pd.concat(pdata_list, ignore_index=True).reset_index(drop=True)

        for i, task_df in enumerate(pdata_list):
            print(i, task_df)
                        
            task_oxy_deoxy = run_matlab_oxy_deoxy(task_df, all_pdata)
            
            # one participant's data is bad; show this here if needed. we drop them later. 
            if (task_oxy_deoxy == -np.inf).sum().sum() > 0 or (task_oxy_deoxy == np.inf).sum().sum() > 0:
                print(f"pid: {pid} - {task_oxy_deoxy.shape}")
                # print(pd.DataFrame({'-inf': (task_oxy_deoxy == -np.inf).sum(), 'inf': (task_oxy_deoxy == np.inf).sum()}))            
                
            meta = task_oxy_deoxy.iloc[0].loc[['MARKER', 'pid']]
            brain_keys = [key for key in task_oxy_deoxy.keys() if 'DSI' in key or 'phi' in key]
                
            #task_oxy_deoxy = task_oxy_deoxy.drop(columns=['time'])        
            
            # apply the butterworth filter
            task_oxy_deoxy.loc[:, brain_keys] = task_oxy_deoxy[brain_keys].apply(butter_bandpass_filter)

            task_oxy_deoxy['MARKER'] = meta['MARKER']
            task_oxy_deoxy['pid'] = meta['pid']
            task_oxy_deoxy['trial_id'] = i

            all_segments.append(task_oxy_deoxy)        

    fdf_oxy_deoxy = pd.concat(all_segments, ignore_index=True)    
    return fdf_oxy_deoxy, all_segments


In [10]:
import pandas as pd
from scipy.signal import butter, filtfilt

# from leon - apply butterworth filter to data
# https://openreview.net/pdf?id=QzNHE7QHhut

# Function to design a Butterworth filter
def butter_bandpass():
    lowcut = 0.001
    highcut = 0.2
    fs = 5.2084
    order = 4

    b, a = butter(order, [lowcut, highcut], fs=fs, btype='band')
    return b, a

# Function to apply a Butterworth filter
def butter_bandpass_filter(data):
    b, a = butter_bandpass()
    return filtfilt(b, a, data)    

In [11]:
def load_fnirs_data():
    dfs = []
    for f in os.listdir('data'):
        pid = f.split('_')[0]
        pdf = pd.read_csv(filepath_or_buffer=os.path.join('data', f), encoding='utf-8-sig')
        pdf['pid'] = pid
        pdf = pdf.drop(columns=[k for k in pdf.columns if 'Hb' in k])
        pdf.loc[:, 'T'] -= pdf['T'].iloc[0]
        dfs.append(pdf)

    df = pd.concat(dfs, ignore_index=True)
    return df

def load_and_preprocess_fnirs_data():
    df = load_fnirs_data()
    df = remove_unnecessary_markers(df)
    df = df.rename(columns={'TIME': 'time'})
    df['time'] = pd.to_datetime(df['time'], unit='ms')  
    return df

In [12]:
def convert_probe_values_to_matlab(df):    
    if df.shape[0] < 10:
        return None, None, None, None, None, None, None, None, None

    A_I = matlab.double(df[['AC-A1', 'AC-A2', 'AC-A21', 'AC-A22']].values.tolist())
    B_I = matlab.double(df[['AC-B1', 'AC-B2', 'AC-B21', 'AC-B22']].values.tolist())
    C_I = matlab.double(df[['AC-C3', 'AC-C4', 'AC-C23', 'AC-C24']].values.tolist())
    D_I = matlab.double(df[['AC-D3', 'AC-D4', 'AC-D23', 'AC-D24']].values.tolist())

    A_PH = matlab.double(df[['PH-A1', 'PH-A2', 'PH-A21', 'PH-A22']].values.tolist())
    B_PH = matlab.double(df[['PH-B1', 'PH-B2', 'PH-B21', 'PH-B22']].values.tolist())
    C_PH = matlab.double(df[['PH-C3', 'PH-C4', 'PH-C23', 'PH-C24']].values.tolist())
    D_PH = matlab.double(df[['PH-D3', 'PH-D4', 'PH-D23', 'PH-D24']].values.tolist())

    t = df['T'] - df['T'].iloc[0]
    t = matlab.double(t.values.tolist())    

    return A_I, B_I, C_I, D_I, A_PH, B_PH, C_PH, D_PH, t


def run_matlab_oxy_deoxy(df, all_pdata):
    
    lamb = matlab.double([830, 690]) 

    outvars = ['I', 'phi', 'O_DSI', 'D_DSI', 'O_DSphi', 'D_DSphi']

    A_I, B_I, C_I, D_I, A_PH, B_PH, C_PH, D_PH, t = convert_probe_values_to_matlab(df)
    A_I_all, B_I_all, C_I_all, D_I_all, A_PH_all, B_PH_all, C_PH_all, D_PH_all, t_all = convert_probe_values_to_matlab(all_pdata)
              
    # Call MATLAB functions
    outputs = eng.getOxyDeoxy(A_I, B_I, A_PH, B_PH, t,
                              A_I_all, B_I_all, A_PH_all, B_PH_all, t_all,
                              lamb, nargout=6)    
    PL = { f"L_{outvars[i]}": np.array(outputs[i]) for i in range(len(outvars)) }        
    PL_OXY_DEOXY = pd.DataFrame({ key: value.flatten() for key, value in PL.items() if 'O_DS' in key or 'D_DS' in key })            

    outputs = eng.getOxyDeoxy(C_I, D_I, C_PH, D_PH, t,
                              C_I_all, D_I_all, C_PH_all, D_PH_all, t_all,
                              lamb, nargout=6)
    PR = { f"R_{outvars[i]}": np.array(outputs[i]) for i in range(len(outvars)) }
    PR_OXY_DEOXY = pd.DataFrame({ key: value.flatten() for key, value in PR.items() if 'O_DS' in key or 'D_DS' in key })        
        
    # to inspect either dataset        
    # plot_data(PL, np.array(t).flatten(), lamb)
    # plot_data(PR, np.array(t).flatten(), lamb)
        
    P_OXY_DEOXY = pd.concat([PL_OXY_DEOXY, PR_OXY_DEOXY], axis=1)
        
    # return original data
    OD_data = pd.concat([df, P_OXY_DEOXY], axis=1)
    OD_data = OD_data.drop(columns=['T'] + [key for key in OD_data if 'PH' in key or 'AC' in key])  

    return OD_data


In [13]:
def remove_unnecessary_markers(fdf):
    # these offsets assume pdf starts at index 0
    # add each new participant here after manual checks. 
    markers_to_remove = {
        'PID-here': ['list of indices to remove - manually determine this! exmample below'],    
        '000': [],
        '002': [], 
        '003': [], 
        '004': [], 
        '005': [], 
        '006': [], 
        '007': [], 
        '008': [], 
        '009': [], 
        '010': [],    
        '011': [], 
        '012': [], 
        '013': [], 
        '014': [], 
        '015': [], 
        '016': [], 
        '017': [], 
        '018': [], 
        '019': [], 
        '020': [], 
        '021': [], 
        '022': [], 
        '023': [], 
        '024': [],
        '025': [], 


    }

    pdfs = []
    for pid, pdf in fdf.groupby('pid'):
        pdf = pdf.reset_index(drop=True)
                                                
        if pid not in markers_to_remove.keys():            
            pdf.loc['TIME'] = pd.to_datetime(pdf['TIME'], unit='ms')
            print(pdf.loc[pdf['MARKER'] != '0', ['MARKER', 'TIME']])           
            raise ValueError(f'need to manually check marker for: {pid}')

        pdf.loc[markers_to_remove[pid], "MARKER"] = "0"
        pdfs.append(pdf)         

    fdf = pd.concat(pdfs).reset_index(drop=True)
    return fdf

Code to graph fnirs data. 

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# Define the lowpass filter function
def matlab_style_lowpass(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data, axis=0)
    return y

In [15]:

def plot_fnirs_data(PX, t, t2, lamb):   
    
    # Sample frequency calculation
    fs = 1 / np.median(np.diff(t))

    # Define x-axis limits
    xl = [t[0], t[-1]]

    # Sample frequency calculation
    fs2 = 1 / np.median(np.diff(t2))

    # Define x-axis limits
    xl2 = [t2[0], t2[-1]]

    # Create figure and subplots
    fig, axs = plt.subplots(3, 2, figsize=(12, 10))
    fig.suptitle('FNIRS Data Visualization')

    # Intensity plots
    for lInd in range(2):  # Assuming 2 lambdas
        I0 = np.mean(PX['I'][:, :, lInd], axis=0)
        offset = np.linspace(-0.075, 0.075, PX['I'].shape[1])
        normalized_I = (PX['I'][:, :, lInd] - I0) / I0
        for i in range(normalized_I.shape[1]):
            normalized_I[:, i] += offset[i]
        filtered_I = matlab_style_lowpass(normalized_I, 0.2, fs)
        axs[lInd, 0].plot(t, filtered_I)
        axs[lInd, 0].set_xlim(xl)
        axs[lInd, 0].set_ylabel(r'$\Delta I/I_0$')
        axs[lInd, 0].set_xlabel('t (sec)')
        axs[lInd, 0].set_title(f'Intensity Data {lamb[0][lInd]} nm')

    # Dual-Slope Intensity
    axs[2, 0].plot(t2, matlab_style_lowpass(PX['O_DSI'] + 0.3, 0.2, fs2), '-r')
    axs[2, 0].plot(t2, matlab_style_lowpass(PX['D_DSI'] - 0.3, 0.2, fs2), '-b')
    axs[2, 0].set_xlim(xl2)
    axs[2, 0].set_ylabel(r'$\Delta$ (\mu M)')
    axs[2, 0].set_xlabel('t (sec)')
    axs[2, 0].set_title('Dual-Slope Intensity')

    # Phase plots
    for lInd in range(2):  # Assuming 2 lambdas
        phi0 = PX['phi'][:, :, lInd]
        offset = np.linspace(-0.015, 0.015, PX['phi'].shape[1])
        normalized_phi = PX['phi'][:, :, lInd] - phi0
        for i in range(normalized_phi.shape[1]):
            normalized_phi[:, i] += offset[i]
        filtered_phi = matlab_style_lowpass(normalized_phi, 0.2, fs)
        axs[lInd, 1].plot(t, filtered_phi)
        axs[lInd, 1].set_xlim(xl)
        axs[lInd, 1].set_ylabel(r'$\Delta \phi$ (rad)')
        axs[lInd, 1].set_xlabel('t (sec)')
        axs[lInd, 1].set_title(f'Phase Data {lamb[0][lInd]} nm')

    # Dual-Slope Phase
    axs[2, 1].plot(t2, matlab_style_lowpass(PX['O_DSphi'] + 0.3, 0.2, fs2), '-r')
    axs[2, 1].plot(t2, matlab_style_lowpass(PX['D_DSphi'] - 0.3, 0.2, fs2), '-b')
    axs[2, 1].set_xlim(xl2)
    axs[2, 1].set_ylabel(r'$\Delta$ (\mu M)')
    axs[2, 1].set_xlabel('t (sec)')
    axs[2, 1].set_title('Dual-Slope Phase')

    # Display the plot
    plt.tight_layout()
    plt.show()